# AgriTwin — Training Monitor (historical)

This notebook was used to watch the cross-domain training runs on an RTX 5060. **All
experiments are complete as of 2026-11-24**; the deployed model and final metrics are in
`backend/app/ml/artifacts/crop_health/metadata.json` and `docs/FINAL_REPORT.md` §12–13.
It is retained as a reproducible monitoring template. To reuse it for a new run, set the
`AGRITWIN_TRAIN_LOG` environment variable to the trainer's stdout file (cell 2).

- **Cell 2** — one-shot status snapshot
- **Cell 3** — auto-refreshing live monitor (runs for ~5 minutes; re-run as needed)
- **Cell 4** — results table once `cross_domain_results.json` is written

> Note: the background trainer's stdout is buffered by the shell, so epoch lines appear in
> bursts rather than instantly. GPU utilisation and process status are live. For a fully
> streamed run, launch the next experiment from this notebook with `python -u ...` (see cell 5).

In [2]:
from pathlib import Path
import json, os, subprocess, time
from datetime import datetime
from IPython.display import clear_output, display, Markdown

ROOT = Path.cwd()
LOG = Path(os.environ.get("AGRITWIN_TRAIN_LOG", str(ROOT / "training.log")))
RESULTS = ROOT / "ml/crop_health/evaluation/cross_domain_results.json"
META = ROOT / "backend/app/ml/artifacts/crop_health/metadata.json"

def ps(cmd: str) -> str:
    return subprocess.run(["powershell", "-NoProfile", "-Command", cmd],
                          capture_output=True, text=True, timeout=30).stdout.strip()

def snapshot(lines: int = 18) -> str:
    proc = ps("Get-CimInstance Win32_Process -Filter \"name='python.exe'\" | "
              "Where-Object {$_.CommandLine -match 'train_field_robust'} | "
              "ForEach-Object { \"pid=$($_.ProcessId) started=$($_.CreationDate)\" }")
    gpu = ps("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader")
    tail = LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:] if LOG.exists() else []
    out = [f"**Time:** {datetime.now():%H:%M:%S}",
           f"**Trainer:** {proc or 'NOT RUNNING'}",
           f"**GPU (util %, memory):** {gpu or 'n/a'}",
           f"**Results file:** {'present' if RESULTS.exists() else 'not yet written'}",
           "", "```", *tail[-8:], "```"]
    return "\n".join(out)

display(Markdown(snapshot()))

**Time:** 22:37:51
**Trainer:** NOT RUNNING
**GPU (util %, memory):** 0 %, 0 MiB
**Results file:** present

```
  Grad-CAM skipped for 1b321015-6e33-4f18-aade-888f4383fe92.jpeg.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 28-500x375.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0004.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0011.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0019.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

Deployed 'two_stage' to C:\Users\arvap\OneDrive\Documents\Desktop\majorp\backend\app\ml\artifacts\crop_health
Full results: C:\Users\arvap\OneDrive\Documents\Desktop\majorp\ml\crop_health\evaluation\cross_domain_results.json
```

In [3]:
# Auto-refreshing live monitor (30 refreshes x 10s = 5 minutes). Re-run when it stops.
for _ in range(30):
    clear_output(wait=True)
    display(Markdown(snapshot()))
    if RESULTS.exists():
        display(Markdown("**Training finished — results are available. Run cell 4.**"))
        break
    time.sleep(10)

**Time:** 22:37:55
**Trainer:** NOT RUNNING
**GPU (util %, memory):** 0 %, 0 MiB
**Results file:** present

```
  Grad-CAM skipped for 1b321015-6e33-4f18-aade-888f4383fe92.jpeg.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 28-500x375.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0004.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0011.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor
  Grad-CAM skipped for 0019.jpg: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

Deployed 'two_stage' to C:\Users\arvap\OneDrive\Documents\Desktop\majorp\backend\app\ml\artifacts\crop_health
Full results: C:\Users\arvap\OneDrive\Documents\Desktop\majorp\ml\crop_health\evaluation\cross_domain_results.json
```

**Training finished — results are available. Run cell 4.**

In [4]:
# Cross-domain summary table (run after training completes).
if not RESULTS.exists():
    print("Results not ready yet.")
else:
    data = json.loads(RESULTS.read_text(encoding="utf-8"))
    print(f"Created: {data['created_at']}  |  architecture: {data['architecture']}\n")
    header = f"{'Experiment':<18}{'PV acc':>9}{'PD acc':>9}{'PV F1':>9}{'PD F1':>9}"
    print(header)
    print("-" * len(header))
    for row in data["summary"]:
        print(f"{row['experiment']:<18}{row.get('pv_accuracy', '-'):>9}"
              f"{row.get('pd_accuracy', '-'):>9}{row.get('pv_f1_macro', '-'):>9}"
              f"{row.get('pd_f1_macro', '-'):>9}")
    if META.exists():
        meta = json.loads(META.read_text(encoding="utf-8"))
        print(f"\nDeployed: {meta.get('version')} | mode={meta.get('training_mode')} "
              f"| amp={meta.get('amp')} | exg={meta.get('exg_suppression')}")

Created: 2026-11-24T15:04:32.130964+00:00  |  architecture: efficientnet_b0

Experiment           PV acc   PD acc    PV F1    PD F1
------------------------------------------------------
baseline_pv          0.9549    0.322   0.9539   0.2177
plantdoc_only         0.312   0.6822   0.2396   0.6721
mixed                0.9511   0.6653   0.9507   0.6563
two_stage            0.8308   0.5551   0.8322   0.5292

Deployed: crop-health-v2.0.0-field-robust | mode=mixed | amp=True | exg=False


### To stream the next run fully live
Unbuffered Python (`-u`) writes each epoch line immediately into the notebook output. For the
optional ExG ablation (do **not** run while the main job is still training):

```python
!python -u ml/crop_health/training/train_field_robust.py --mode two_stage --epochs 12 --amp \
    --exg-suppression --exg-strength 0.8 \
    --artifact-dir backend/app/ml/artifacts/crop_health_exg \
    --eval-dir ml/crop_health/evaluation/exg
```